# Measure any retrieval endpoint against MisakaNet's 267-query set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ikalus1988/MisakaNet/blob/main/docs/notebooks/reproduce-retrieval.ipynb)

This notebook has one job: take the published query set, point it at **any** MCP-compatible search
endpoint, and print R@1 / R@3 / R@5 by language and overlap.

It is not a rerun of our numbers for its own sake — it is the measurement we publish, made
applicable to *someone else's* endpoint. The query set and every published run are at
<https://data.misakanet.org/latest.json>.

Nothing here needs credentials, a clone, or a GPU. Runtime: about 6 minutes, almost all of it pacing
against the endpoint's burst guard.


In [ ]:
# The release this notebook was last checked against. Bump it when the query set changes.
VERSION = "2026-10-28-4f39aa0"
BASE = "https://data.misakanet.org"

import json, time, urllib.request

def get(url, timeout=60):
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "misakanet-notebook/1.0"}), timeout=timeout) as r:
        return r.read()

manifest = json.loads(get(f"{BASE}/v{VERSION}/manifest.json"))
queries = [json.loads(line) for line in get(f"{BASE}/v{VERSION}/benchmarks/retrieval-l1-267-queries.jsonl").decode().splitlines() if line.strip()]
print(f"{manifest['version']}: {len(queries)} queries, licenses {[l['id'] for l in manifest['licenses'].values()]}")
print(queries[0])

## Point it at an endpoint

The default is the public MisakaNet endpoint, so a fresh Run reproduces the live arm. Replace
`ENDPOINT` with yours; it only has to speak MCP over JSON-RPC and expose a search tool with
`query` / `top`.

Two knobs exist because both were needed in practice: **pacing** (the anonymous read path has a burst
guard per address, and a refusal is a speed limit, not a quota) and **retries** (a dropped connection
must never be scored as "no result" — that is how a flaky network becomes a fake recall number).

In [ ]:
ENDPOINT = "https://misakanet.org/mcp"   # <- your endpoint
TOOL = "misakanet_search"               # <- the search tool's name
TOP = 5
DELAY = 2.0                             # seconds between queries (~28/min)
ATTEMPTS = 4

def mcp_search(query):
    payload = {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
               "params": {"name": TOOL, "arguments": {"query": query, "top": TOP}}}
    last = None
    for attempt in range(ATTEMPTS):
        try:
            req = urllib.request.Request(
                ENDPOINT, data=json.dumps(payload).encode(),
                headers={"Content-Type": "application/json", "Accept": "application/json",
                         "MCP-Protocol-Version": "2025-06-18", "Origin": "https://misakanet.org",
                         "User-Agent": "misakanet-notebook/1.0"})
            with urllib.request.urlopen(req, timeout=40) as r:
                body = json.loads(r.read())
            result = body.get("result") or {}
            return result.get("structuredContent") or json.loads((result.get("content") or [{}])[0]["text"])
        except Exception as exc:          # noqa: BLE001 - transport faults are retried, not scored
            last = exc
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"gave up on {query!r}: {last}")

In [ ]:
rows, failures = [], 0
for i, q in enumerate(queries, 1):
    try:
        payload = mcp_search(q["query"])
    except RuntimeError:
        failures += 1
        continue
    ids = [r.get("id") for r in (payload.get("results") or [])]
    rank = ids.index(q["lesson_id"]) + 1 if q["lesson_id"] in ids else None
    rows.append({"lesson_id": q["lesson_id"], "lang": q["lang"], "overlap": q["overlap_bucket"],
                 "rank": rank, "no_match": bool(payload.get("no_match"))})
    if i % 25 == 0:
        print(f"{i}/{len(queries)} done, {failures} transport failure(s)")
    time.sleep(DELAY)

failed = [r for r in rows if r["rank"] is None]
print(f"\nscored {len(rows)} of {len(queries)}; {failures} transport failure(s) excluded from every rate")

## Score

`no_match` here means "the target was not in the top-5". That is not the same quantity as the
endpoint's own `no_match` (which means it returned nothing), and the two must never be compared —
the published reports say the same thing for the same reason.

In [ ]:
def rate(subset, k):
    return sum(1 for r in subset if r["rank"] and r["rank"] <= k) / len(subset) if subset else 0.0

print(f"{'bucket':12s} {'n':>4}  {'R@1':>6} {'R@3':>6} {'R@5':>6} {'miss':>6}")
for lang in sorted({r['lang'] for r in rows}):
    for overlap in sorted({r['overlap'] for r in rows if r['lang'] == lang}):
        sub = [r for r in rows if r['lang'] == lang and r['overlap'] == overlap]
        print(f"{lang+'/'+overlap:12s} {len(sub):4d}  {rate(sub,1):6.3f} {rate(sub,3):6.3f} "
              f"{rate(sub,5):6.3f} {1-rate(sub,5):6.3f}")
print(f"{'all':12s} {len(rows):4d}  {rate(rows,1):6.3f} {rate(rows,3):6.3f} {rate(rows,5):6.3f} "
      f"{1-rate(rows,5):6.3f}")

## What this does and does not show

- **Does:** how well *this* endpoint, on *this* query set, returns the lesson each query was written
  about. Comparable across endpoints and over time, because the query set is versioned and immutable.
- **Does not:** whether the endpoint is useful to a human. The queries are generated from the target
  lessons, so they share vocabulary with the answer; absolute rates flatter any retriever.
- **Does not:** separate ranking quality from corpus coverage. A miss can be a stale target, a
  poorly written query, or a real ranking failure, and this notebook cannot tell them apart.

Published runs, with method and limitations for each: <https://github.com/Ikalus1988/MisakaNet/tree/main/docs/benchmarks>.
